# 02 — Working-set analysis

The key experiment: latency vs model footprint scaling curves,
with piecewise regression breakpoints overlaid.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from analysis.piecewise_regression import find_breakpoint, chow_test

plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
master = pd.read_csv('data/processed/master.csv')
print(master.shape)
master.head()

In [ ]:
# Inspect working-set sweep data
sweep = master[master['model_name'].str.contains('sweep_', na=False)].copy()
print(f'Sweep rows: {len(sweep)}')
print(sweep.groupby(['precision','compute_unit'])['model_size_mb'].describe())

In [ ]:
# Latency vs footprint — THE KEY FIGURE
cu = 'cpuAndNeuralEngine'
sub = sweep[sweep['compute_unit'] == cu]

colors = {'fp32': '#185FA5', 'fp16': '#1D9E75', 'int8_linear': '#D85A30'}
fig, ax = plt.subplots()
for prec, group in sub.groupby('precision'):
    g = group.sort_values('model_size_mb')
    ax.plot(g['model_size_mb'], g['median_us'], 'o-',
            color=colors.get(prec, 'gray'), label=prec, linewidth=2, markersize=5)

    # Overlay breakpoint
    x, y = g['model_size_mb'].values, g['median_us'].values
    if len(x) >= 6:
        res = find_breakpoint(x, y)
        ax.axvline(res['breakpoint_mb'], color=colors.get(prec,'gray'),
                   linestyle=':', linewidth=1.5, alpha=0.7)
        ax.text(res['breakpoint_mb'] + 0.3, y.max() * 0.95,
                f"{prec}\nbp={res['breakpoint_mb']:.1f}MB", fontsize=7, color=colors.get(prec,'gray'))

ax.set_xlabel('Model footprint (MB)')
ax.set_ylabel('Median latency (µs)')
ax.set_title(f'Latency vs footprint — {cu}')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Chow test results for each precision
for prec, group in sub.groupby('precision'):
    g = group.sort_values('model_size_mb')
    x, y = g['model_size_mb'].values, g['median_us'].values
    if len(x) < 6: continue
    bp_res = find_breakpoint(x, y)
    chow = chow_test(x, y, bp_res['breakpoint_mb'])
    sig = 'SIGNIFICANT' if chow.get('significant_005') else 'not significant'
    print(f'{prec:20s}  bp={bp_res["breakpoint_mb"]:.1f}MB  slope_ratio={bp_res["slope_ratio"]}  Chow_p={chow.get("chow_p")}  [{sig}]')

In [ ]:
# INT8 / FP32 breakpoint ratio (hypothesis: should be ~2.0)
bp_fp32 = None
bp_int8 = None
for prec, group in sub.groupby('precision'):
    g = group.sort_values('model_size_mb')
    x, y = g['model_size_mb'].values, g['median_us'].values
    if len(x) < 6: continue
    bp = find_breakpoint(x, y)['breakpoint_mb']
    if prec == 'fp32': bp_fp32 = bp
    if prec == 'int8_linear': bp_int8 = bp
if bp_fp32 and bp_int8:
    ratio = bp_int8 / bp_fp32
    print(f'INT8 breakpoint / FP32 breakpoint = {ratio:.2f}×')
    print(f'Expected if cache dominates: ~2.0×  (INT8 tensors are ~½ the size of FP32)')